# Etape 3 : Classification supervisee - Prediction du statut COVID-19

Dans les etapes precedentes, nous avons :
1. **Etape 1** : Charge et explore les donnees brutes (couche Bronze -> Silver)
2. **Etape 2** : Nettoye et standardise les donnees (couche Silver -> Gold)

Jusqu'ici, notre travail portait sur le **clustering non supervise** (K-Means, DBSCAN) pour identifier des profils de patients sans utiliser le diagnostic COVID.

Dans cette etape, nous changeons d'approche : nous passons a l'**apprentissage supervise**. L'objectif est de construire un modele capable de **predire si un patient est COVID+ ou COVID-** a partir de ses caracteristiques cliniques (age, comorbidites, etc.).

**Pourquoi l'apprentissage supervise ?**
- Contrairement au clustering, ici **on dispose de la reponse** (`Label_Resultat_COVID`). On peut donc entrainer un modele a reconnaitre les patterns qui distinguent un patient positif d'un negatif.
- Cela permet ensuite de **predire le resultat pour un nouveau patient** dont on ne connait pas encore le statut COVID.

**Plan de cette etape :**
1. Chargement et preparation des donnees Gold
2. Analyse de la distribution de la cible (equilibre des classes)
3. Recherche metrique d'hyperparametres (GridSearch/RandomSearch) pour les modeles
4. Entrainement et comparaison des algorithmes optimises
5. Evaluation detaillee (metriques, courbes ROC, matrices de confusion)
6. Selection et sauvegarde du **meilleur modele** pour le questionnaire patient

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
import os

# Scikit-learn : outils de Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report, average_precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Algorithmes de Boosting (etat de l'art)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Pour sauvegarder le modele final
import joblib

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

print('Librairies chargees avec succes')

## 1. Chargement et preparation des donnees Gold

On reprend le dataset de la couche Gold (`covid_clustering_ready.csv`) qui a ete prepare a l'etape 2. Rappelons ses caracteristiques :
- Toutes les variables sont **binaires (0 ou 1)**, y compris l'age qui a ete discretise en 4 tranches
- Les comorbidites ont ete recodees de `1/2` (SISVER) vers `1/0` (standard)
- La colonne cible `Label_Resultat_COVID` contient : `1` = COVID+ et `2` = COVID-

**Transformation necessaire :** Pour la classification supervisee, il est conventionnel d'utiliser `0` et `1` pour la cible. On convertit donc `2 -> 0` (COVID-) et on garde `1 -> 1` (COVID+).

In [ ]:
# -- 1. Chargement des donnees Gold --
GOLD_PATH = '../../data/layer_gold_data_model/covid_clustering_ready.csv'
df = pd.read_csv(GOLD_PATH)

print(f'Dataset charge : {df.shape[0]:,} patients, {df.shape[1]} colonnes.')
print(f'\nColonnes disponibles :')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
# -- Separation features / cible --
TARGET = 'Label_Resultat_COVID'

X = df.drop(columns=[TARGET])
y = df[TARGET].replace(2, 0)  # 2 (COVID-) -> 0, 1 (COVID+) reste 1

print(f'Features (X) : {X.shape[0]:,} lignes x {X.shape[1]} colonnes')
print(f'Cible (y)    : {y.shape[0]:,} valeurs')
print(f'\nApercu des features :')
X.head(3)

## 2. Analyse de la distribution de la cible

Avant de choisir nos algorithmes, il est **essentiel** de verifier l'equilibre des classes. Un dataset desequilibre (par exemple 95% negatif / 5% positif) necessite des strategies specifiques (sur-echantillonnage, ponderation, etc.).

**Pourquoi c'est important ?**
- Si 90% des patients sont COVID-, un modele naif qui predit toujours "negatif" aurait 90% d'accuracy mais serait completement inutile pour detecter les cas positifs.
- C'est pourquoi on utilisera le **F1-Score** comme metrique principale (pas l'accuracy), car il prend en compte a la fois la Precision et le Recall.

In [ ]:
# -- Visualisation de la distribution --
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Comptage
counts = y.value_counts()
labels = ['COVID- (0)', 'COVID+ (1)']
colors = ['#2ecc71', '#e74c3c']

# Barplot
axes[0].bar(labels, counts.sort_index().values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribution des classes', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Nombre de patients')
for i, v in enumerate(counts.sort_index().values):
    axes[0].text(i, v + 1500, f'{v:,}', ha='center', fontweight='bold', fontsize=11)

# Pie chart
axes[1].pie(counts.sort_index().values, labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Proportion des classes', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../../data/layer_gold_data_model/distribution_classes.png', dpi=150, bbox_inches='tight')
plt.show()

ratio = counts[1] / counts[0]
print(f'\nRatio COVID+ / COVID- : {ratio:.2f}')
print(f'Le dataset est legerement desequilibre ({counts[1]/len(y)*100:.1f}% positifs vs {counts[0]/len(y)*100:.1f}% negatifs).')
print('-> Ce desequilibre est modere, on peut travailler sans sur-echantillonnage.')
print('-> Cependant, on utilisera le F1-Score comme metrique principale plutot que l\'accuracy.')

## 3. Decoupage Train / Test

On separe le dataset en **80% entrainement** et **20% test** :
- Le set d'**entrainement** sert a apprendre les patterns.
- Le set de **test** sert a evaluer la performance sur des donnees jamais vues (simulation de nouveaux patients).

**Stratification (`stratify=y`)** : On s'assure que la proportion COVID+/COVID- est **identique** dans le train et le test. Sans cela, par malchance, on pourrait avoir 50% COVID+ dans le train et 30% dans le test, ce qui biaiserait l'evaluation.

**`random_state=42`** : Graine fixe pour la reproductibilite. N'importe qui relancant ce notebook obtiendra exactement les memes resultats.

In [ ]:
# -- Split stratifie 80/20 --
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Preserve le ratio COVID+/COVID-
)

print(f'Entrainement : {X_train.shape[0]:,} patients ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test         : {X_test.shape[0]:,} patients ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'\nVerification de la stratification :')
print(f'  Train -> COVID+ : {y_train.mean()*100:.1f}%')
print(f'  Test  -> COVID+ : {y_test.mean()*100:.1f}%')
print(f'  Total -> COVID+ : {y.mean()*100:.1f}%')
print('\nLes proportions sont identiques (stratification reussie).')

## 4. Recherche Metrique d'Hyperparametres (Solution Metrique)

Au lieu de fixer des hyperparametres au hasard, nous allons utiliser une approche scientifique : la recherche d'hyperparametres via **Validation Croisee (Cross-Validation)** en utilisant des metriques objectives (ici, le **F1-Score**).

Comme le dataset est tres grand (260 000 patients), explorer toutes les combinaisons prendrait des jours. Nous allons :
1. Prendre un **sous-echantillon** representatif (30 000 patients) du set d'entrainement.
2. Appliquer un **RandomizedSearchCV** (recherche aleatoire sur une grille) ou **GridSearchCV**.
3. Trouver les meilleurs parametres pour le Random Forest, XGBoost et LightGBM.
4. La Regression Logistique sera optimisée via GridSearchCV sur son parametre C.

In [ ]:
# -- Creation du sous-echantillon pour la recherche --
SAMPLE_SIZE_TUNING = 30000
indices_tuning = np.random.RandomState(42).choice(len(X_train), size=SAMPLE_SIZE_TUNING, replace=False)

X_train_tune = X_train.iloc[indices_tuning]
y_train_tune = y_train.iloc[indices_tuning]

print(f'Taille de l\'echantillon pour le tuning : {X_train_tune.shape[0]} patients')
print(f'Ratio COVID+ dans l\'echantillon : {y_train_tune.mean()*100:.1f}% (identique au dataset complet)')

In [ ]:
# -- Definition des grilles de recherche --
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Dictionnaire pour stocker les meilleurs parametres trouves
best_params_dict = {}

print('=' * 60)
print('LANCEMENT DE LA RECHERCHE METRIQUE D\'HYPERPARAMETRES')
print('=' * 60)

# 1. Regression Logistique
print('\n1. Optimisation Regression Logistique...')
param_grid_lr = {'C': [0.01, 0.1, 1.0, 10.0], 'solver': ['lbfgs']}
grid_lr = GridSearchCV(LogisticRegression(max_iter=500, random_state=42, class_weight='balanced'), param_grid_lr, cv=cv_strategy, scoring='f1', n_jobs=-1)
grid_lr.fit(X_train_tune, y_train_tune)
best_params_dict['Regression Logistique'] = grid_lr.best_params_
print(f'   Meilleurs parametres : {grid_lr.best_params_}')
print(f'   Meilleur F1-Score : {grid_lr.best_score_:.4f}')

# 2. Random Forest
print('\n2. Optimisation Random Forest...')
param_dist_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10]
}
rs_rf = RandomizedSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced'), param_dist_rf, n_iter=15, cv=cv_strategy, scoring='f1', random_state=42, n_jobs=-1)
rs_rf.fit(X_train_tune, y_train_tune)
best_params_dict['Random Forest'] = rs_rf.best_params_
print(f'   Meilleurs parametres : {rs_rf.best_params_}')
print(f'   Meilleur F1-Score : {rs_rf.best_score_:.4f}')

# 3. XGBoost
print('\n3. Optimisation XGBoost...')
param_dist_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.01, 0.1, 0.2]
}
rs_xgb = RandomizedSearchCV(XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=1.57), param_dist_xgb, n_iter=15, cv=cv_strategy, scoring='f1', random_state=42, n_jobs=-1)
rs_xgb.fit(X_train_tune, y_train_tune)
best_params_dict['XGBoost'] = rs_xgb.best_params_
print(f'   Meilleurs parametres : {rs_xgb.best_params_}')
print(f'   Meilleur F1-Score : {rs_xgb.best_score_:.4f}')

# 4. LightGBM
print('\n4. Optimisation LightGBM...')
param_dist_lgb = {
    'n_estimators': [100, 200],
    'max_depth': [-1, 10, 20],
    'learning_rate': [0.01, 0.1, 0.2],
    'num_leaves': [31, 50, 100]
}
rs_lgb = RandomizedSearchCV(LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced'), param_dist_lgb, n_iter=15, cv=cv_strategy, scoring='f1', random_state=42, n_jobs=-1)
rs_lgb.fit(X_train_tune, y_train_tune)
best_params_dict['LightGBM'] = rs_lgb.best_params_
print(f'   Meilleurs parametres : {rs_lgb.best_params_}')
print(f'   Meilleur F1-Score : {rs_lgb.best_score_:.4f}')

print('\n' + '=' * 60)
print('RECHERCHE TERMINEE')
print('=' * 60)

## 5. Configuration des modeles optimises

Nous construisons maintenant les 6 modeles a comparer, en injectant les hyperparametres trouves a l'etape precedente.

In [ ]:
# -- Definition des modeles optimises --
modeles = {
    'Regression Logistique': LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced',
        **best_params_dict['Regression Logistique']
    ),
    'KNN (K=5)': KNeighborsClassifier(
        n_neighbors=5, # KNN est garde avec k=5 pour benchmark de base
        metric='euclidean'
    ),
    'Random Forest': RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        class_weight='balanced',
        **best_params_dict['Random Forest']
    ),
    'SVM (RBF)': SVC(
        kernel='rbf',
        probability=True,
        class_weight='balanced',
        random_state=42
    ),
    'XGBoost': XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        scale_pos_weight=1.57,
        **best_params_dict['XGBoost']
    ),
    'LightGBM': LGBMClassifier(
        random_state=42,
        verbose=-1,
        class_weight='balanced',
        **best_params_dict['LightGBM']
    )
}

print(f'{len(modeles)} modeles prets a etre entraines sur le dataset complet.')

## 6. Entrainement et evaluation sur le jeu de test complet

Maintenant que nous avons les meilleurs parametres, nous entrainons chaque modele sur **l'ensemble des donnees d'entrainement (80%)** et nous l'evaluons sur le **jeu de test (20%)** qui n'a jamais ete vu lors de l'etape de tuning.

**Note sur le SVM** : Entrainer un SVM sur 200 000 lignes prendrait des heures. Pour ce modele specifique, nous limiterons l'entrainement a 20 000 patients.

In [ ]:
# -- Entrainement et evaluation de chaque modele --
resultats = []
modeles_entraines = {}
predictions = {}
probabilites = {}
seuils_optimaux = {}

SVM_SAMPLE_SIZE = 20000

print('=' * 60)
print('ENTRAINEMENT FINAL ET EVALUATION')
print('=' * 60)

for nom, modele in modeles.items():
    print('\n' + '-' * 60)
    print(f'> {nom}')
    print('-' * 60)
    
    if nom == 'SVM (RBF)':
        print(f'  [!] SVM : sous-echantillonnage a {SVM_SAMPLE_SIZE:,} patients')
        indices = np.random.RandomState(42).choice(len(X_train), size=SVM_SAMPLE_SIZE, replace=False)
        X_train_model = X_train.iloc[indices]
        y_train_model = y_train.iloc[indices]
    else:
        X_train_model = X_train
        y_train_model = y_train
    
    print('  Entrainement en cours...', end=' ')
    t0 = time.time()
    modele.fit(X_train_model, y_train_model)
    t_train = time.time() - t0
    print(f'[{t_train:.1f}s]')
    
    print('  Evaluation...', end=' ')
    y_proba = modele.predict_proba(X_test)[:, 1]
    
    # --- Threshold Tuning pour maximiser le F1-score ---
    best_threshold = 0.5
    best_f1_thresh = 0.0
    for thresh in np.arange(0.1, 0.9, 0.05):
        thresh_pred = (y_proba >= thresh).astype(int)
        thresh_f1 = f1_score(y_test, thresh_pred)
        if thresh_f1 > best_f1_thresh:
            best_f1_thresh = thresh_f1
            best_threshold = thresh
    
    y_pred = (y_proba >= best_threshold).astype(int)
    print(f'OK (Seuil optimal trouve : {best_threshold:.2f})')
    
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    
    print(f'  Resultats sur le Test : F1-Score = {f1:.4f} | PR-AUC = {pr_auc:.4f}')
    
    resultats.append({
        'Modele': nom,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'AUC-ROC': auc,
        'PR-AUC': pr_auc,
        'Temps entrainement (s)': round(t_train, 1)
    })
    modeles_entraines[nom] = modele
    predictions[nom] = y_pred
    probabilites[nom] = y_proba
    seuils_optimaux[nom] = best_threshold

print('\n' + '=' * 60)
print('PROCESSUS TERMINE')
print('=' * 60)

## 7. Tableau comparatif des resultats

Voici la synthese de tous les modeles. Le tableau est trie par **F1-Score decroissant** (notre metrique principale).

In [ ]:
# -- Tableau comparatif --
df_resultats = pd.DataFrame(resultats)
df_resultats = df_resultats.sort_values('F1-Score', ascending=False).reset_index(drop=True)
df_resultats.index = df_resultats.index + 1
df_resultats.index.name = 'Rang'

df_affichage = df_resultats.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'PR-AUC']:
    df_affichage[col] = df_affichage[col].apply(lambda x: f'{x:.4f}')

print('\n[Bar] CLASSEMENT DES MODELES (trie par F1-Score)\n')
display(df_affichage)

## 8. Visualisation comparative des performances

On genere deux types de graphiques pour comparer visuellement les modeles :
1. **Barres groupees** : chaque metrique cote a cote pour chaque modele
2. **Courbes ROC** : visualise la capacite de discrimination de chaque modele

In [ ]:
# -- Graphique 1 : Barres comparatives des metriques --
fig, ax = plt.subplots(figsize=(14, 6))
metriques = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'PR-AUC']
x = np.arange(len(df_resultats))
width = 0.12
colors_bar = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c', '#9b59b6', '#f1c40f']

for i, metrique in enumerate(metriques):
    vals = df_resultats[metrique].values
    ax.bar(x + i * width, vals, width, label=metrique, color=colors_bar[i], edgecolor='white')

ax.set_xlabel('Modele', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparaison des performances par modele', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(df_resultats['Modele'], rotation=20, ha='right', fontsize=10)
ax.legend(loc='lower right', fontsize=9)
ax.set_ylim(0.4, 1.0)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../../data/layer_gold_data_model/comparaison_metriques.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# -- Graphique 2 : Courbes ROC --
fig, ax = plt.subplots(figsize=(8, 8))
colors_roc = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c', '#9b59b6', '#1abc9c']

for i, (nom, y_proba) in enumerate(probabilites.items()):
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_val = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, color=colors_roc[i], lw=2, label=f'{nom} (AUC={auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Aleatoire (AUC=0.500)')
ax.set_xlabel('Taux de Faux Positifs (FPR)', fontsize=12)
ax.set_ylabel('Taux de Vrais Positifs (TPR / Recall)', fontsize=12)
ax.set_title('Courbes ROC - Comparaison des modeles', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../../data/layer_gold_data_model/courbes_roc.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Matrices de confusion

La matrice de confusion montre, pour chaque modele, comment les predictions se repartissent :
- **Vrais Negatifs (VN)** : correctement predits COVID-
- **Faux Positifs (FP)** : predits COVID+ alors qu'ils sont COVID- (fausse alerte)
- **Faux Negatifs (FN)** : predits COVID- alors qu'ils sont COVID+ (cas rates - **le plus dangereux**)
- **Vrais Positifs (VP)** : correctement predits COVID+

In [ ]:
# -- Matrices de confusion pour chaque modele --
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for i, (nom, y_pred) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=axes[i],
                xticklabels=['COVID-', 'COVID+'],
                yticklabels=['COVID-', 'COVID+'],
                annot_kws={'size': 13})
    axes[i].set_title(nom, fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Vrai label')
    axes[i].set_xlabel('Prediction')

plt.suptitle('Matrices de confusion - Tous les modeles', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../../data/layer_gold_data_model/matrices_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Variables les plus impactantes pour la prediction

Comprendre **quelles variables influencent le plus** la prediction est crucial. Nous analysons l'importance des variables de 3 facons complementaires :

### Methode 1 : Importance des features (modeles a base d'arbres)
### Methode 2 : Coefficients de la Regression Logistique
### Methode 3 : Correlation avec la cible

In [ ]:
# -- Identification du meilleur modele --
meilleur_nom = df_resultats.iloc[0]['Modele']
meilleur_modele = modeles_entraines[meilleur_nom]
meilleur_f1 = df_resultats.iloc[0]['F1-Score']

print(f'[Winner] MEILLEUR MODELE : {meilleur_nom}')
print(f'   F1-Score = {meilleur_f1:.4f}')
print(f'\n-- Rapport de classification detaille --\n')
print(classification_report(
    y_test, predictions[meilleur_nom],
    target_names=['COVID-', 'COVID+']
))

In [ ]:
# -- 10a. Importance des features - Meilleur modele --
if hasattr(meilleur_modele, 'feature_importances_'):
    importances = meilleur_modele.feature_importances_
    feature_names = X.columns
    indices = np.argsort(importances)[::-1]
    
    fig, ax = plt.subplots(figsize=(12, 7))
    colors_feat = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(indices)))
    
    bars = ax.barh(range(len(indices)), importances[indices], color=colors_feat, edgecolor='white')
    ax.set_yticks(range(len(indices)))
    ax.set_yticklabels([feature_names[i] for i in indices], fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel('Importance', fontsize=12)
    ax.set_title(f'Importance des variables - {meilleur_nom} (Meilleur modele)', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    for i_bar, (idx, val) in enumerate(zip(indices, importances[indices])):
        ax.text(val + 0.002, i_bar, f'{val:.3f}', va='center', fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../../data/layer_gold_data_model/importance_features_meilleur.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'\n[Bar] Classement des variables par importance ({meilleur_nom}) :')
    for rank, idx in enumerate(indices, 1):
        barre = '#' * int(importances[idx] * 50)
        print(f'  {rank:2d}. {feature_names[idx]:45s} {importances[idx]:.4f} {barre}')
else:
    print(f'Note : {meilleur_nom} ne fournit pas feature_importances_.')

In [ ]:
# -- 10c. Coefficients de la Regression Logistique --
lr_model = modeles_entraines.get('Regression Logistique')
if lr_model is not None and hasattr(lr_model, 'coef_'):
    coefs = lr_model.coef_[0]
    feature_names = X.columns
    indices_abs = np.argsort(np.abs(coefs))[::-1]
    
    fig, ax = plt.subplots(figsize=(12, 7))
    colors_coef = ['#e74c3c' if coefs[i] > 0 else '#3498db' for i in indices_abs]
    
    bars = ax.barh(range(len(indices_abs)), coefs[indices_abs], color=colors_coef, edgecolor='white')
    ax.set_yticks(range(len(indices_abs)))
    ax.set_yticklabels([feature_names[i] for i in indices_abs], fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel('Coefficient (impact sur la prediction COVID+)', fontsize=12)
    ax.set_title('Coefficients de la Regression Logistique', fontsize=14, fontweight='bold')
    ax.axvline(x=0, color='black', linewidth=0.8, linestyle='-')
    ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../../data/layer_gold_data_model/coefficients_logistique.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print('\n[List] Interpretation des coefficients :')
    for idx in indices_abs:
        sens = 'Augmente risque' if coefs[idx] > 0 else 'Diminue risque'
        print(f'  {feature_names[idx]:45s} coef = {coefs[idx]:+.4f}  ({sens})')
else:
    print('La Regression Logistique n\'est pas disponible.')

## 11. Sauvegarde du meilleur modele

On sauvegarde le modele le plus performant avec `joblib` pour pouvoir le reutiliser dans le **questionnaire patient** (Etape 4) sans avoir a le reentrainer.

In [ ]:
# -- Sauvegarde du modele et des metadonnees --
model_dir = '../../data/layer_gold_data_model/modele_final'
os.makedirs(model_dir, exist_ok=True)

model_path = f'{model_dir}/meilleur_modele.joblib'
joblib.dump(meilleur_modele, model_path)

metadata = {
    'nom_modele': meilleur_nom,
    'seuil_optimal': float(seuils_optimaux[meilleur_nom]),
    'features': list(X.columns),
    'metriques': {
        'accuracy': float(df_resultats.iloc[0]['Accuracy']),
        'precision': float(df_resultats.iloc[0]['Precision']),
        'recall': float(df_resultats.iloc[0]['Recall']),
        'f1_score': float(df_resultats.iloc[0]['F1-Score']),
        'auc_roc': float(df_resultats.iloc[0]['AUC-ROC']),
        'pr_auc': float(df_resultats.iloc[0]['PR-AUC'])
    },
    'description_features': {
        'Sexe': '0=Femme, 1=Homme',
        'Type_de_patient': '0=Ambulatoire, 1=Hospitalise',
        'Pneumonie': '0=Non, 1=Oui',
        'Diabète': '0=Non, 1=Oui',
        'Bronchopneumopathie_chronique_obstructive': '0=Non, 1=Oui',
        'Asthme': '0=Non, 1=Oui',
        'Immunosuppression': '0=Non, 1=Oui',
        'Hypertension': '0=Non, 1=Oui',
        'Autre_comorbidité': '0=Non, 1=Oui',
        'Maladie_cardiovasculaire': '0=Non, 1=Oui',
        'Obésité': '0=Non, 1=Oui',
        'Insuffisance_rénale_chronique': '0=Non, 1=Oui',
        'Tabagisme': '0=Non, 1=Oui',
        'Tranche_Age_tranche_age_enfant': '1 si age 0-17 ans',
        'Tranche_Age_tranche_age_jeune': '1 si age 18-39 ans',
        'Tranche_Age_tranche_age_adulte': '1 si age 40-59 ans',
        'Tranche_Age_tranche_age_senior': '1 si age 60+ ans'
    }
}

meta_path = f'{model_dir}/metadata.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f'Modele sauvegarde : {model_path}')
print(f'Metadonnees sauvegardees : {meta_path}')
print(f'\n[Winner] Modele retenu : {meilleur_nom} (F1-Score = {meilleur_f1:.4f})')